# Coffee Leaf Disease Classifier - Visual Testing
This notebook demonstrates how to load the trained ResNet-18 model, display sample leaf images using `matplotlib.pyplot`, and classify them.

In [4]:
%pip install torch torchvision matplotlib Pillow

^C
Note: you may need to restart the kernel to use updated packages.


  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached pillow-13.0.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl.metadata (142 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.4-cp312-


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [9]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as pyplot

CLASS_NAMES = ['healthy', 'rust', 'phoma', 'red_spider_mite']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

ImportError: DLL load failed while importing _C: The specified module could not be found.

In [ ]:
# 1. Define Model Architecture and Load Trained Weights
def build_model(num_classes=4):
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_model(num_classes=4)
# Check checkpoint location
ckpt_path = 'model/best_model.pth' if os.path.exists('model/best_model.pth') else 'checkpoints/best_model.pth'
checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f'Model loaded successfully from Epoch {checkpoint.get("epoch", "unknown")}!')

### Single Sample Prediction (Classroom Style)
Display the leaf image and classify it, just like in class:

In [ ]:
# Choose a test image path
sample_image_path = 'final dataset/test/rust/1200_321.jpg'

# Load image with PIL
image_data = Image.open(sample_image_path).convert('RGB')

# Add a batch dimension to use it with the model
image_tensor = transform(image_data).unsqueeze(0).to(device)

# Model prediction
with torch.no_grad():
    prediction = model(image_tensor)
    probabilities = F.softmax(prediction, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

print('Predicted class:', CLASS_NAMES[pred_idx])
print(f'Confidence: {probabilities[pred_idx]*100:.2f}%')

# Display image using pyplot
pyplot.figure(figsize=(5, 5))
pyplot.imshow(image_data)
pyplot.title(f'Predicted class: {CLASS_NAMES[pred_idx]}', fontsize=12, pad=8)
pyplot.show()

### Multi-Sample Diagnostic Grid (4 Classes)
Picks 1 random leaf from each of the 4 test folders and classifies them:

In [ ]:
test_dir = 'final dataset/test'
fig, axes = pyplot.subplots(1, 4, figsize=(16, 4.5))

for idx, class_name in enumerate(CLASS_NAMES):
    class_folder = os.path.join(test_dir, class_name)
    files = [f for f in os.listdir(class_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    chosen_file = random.choice(files)
    img_path = os.path.join(class_folder, chosen_file)
    
    raw_img = Image.open(img_path).convert('RGB')
    tensor = transform(raw_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(tensor)
        probs = F.softmax(out, dim=1)[0]
        pred_i = torch.argmax(probs).item()
        
    pred_cls = CLASS_NAMES[pred_i]
    conf = probs[pred_i].item() * 100
    color = 'green' if pred_cls == class_name else 'red'
    
    axes[idx].imshow(raw_img)
    axes[idx].set_title(f'True: {class_name}\nPredicted class: {pred_cls}\n({conf:.1f}%)', 
                        color=color, fontweight='bold', fontsize=11)
    axes[idx].tick_params(labelsize=8)

pyplot.tight_layout()
pyplot.show()